In [1]:
# =============================================================================
# STEP 0: Force reload modules (run this first after code changes!)
# =============================================================================
import importlib
import src.data_loader
import src.features
import src.models.base
import src.models.goals
import src.models.assists
import src.models.minutes
import src.models.defcon
import src.models.clean_sheet
import src.models.bonus
import src.models.cards
import src.models.saves
import src.pipeline
import src.viz

importlib.reload(src.data_loader)
importlib.reload(src.features)
importlib.reload(src.models.base)
importlib.reload(src.models.goals)
importlib.reload(src.models.assists)
importlib.reload(src.models.minutes)
importlib.reload(src.models.defcon)
importlib.reload(src.models.clean_sheet)
importlib.reload(src.models.bonus)
importlib.reload(src.models.cards)
importlib.reload(src.models.saves)
importlib.reload(src.pipeline)
importlib.reload(src.viz)

print("All modules reloaded.")

All modules reloaded.


In [2]:
# =============================================================================
# STEP 1: Update Data (optional - only if you need fresh gameweek data)
# =============================================================================
#!python scrape_update_data.py --gameweek 29
# !python scrape_update_data.py --auto

In [3]:
# =============================================================================
# STEP 2: Run the Pipeline
# =============================================================================
from src.pipeline import FPLPipeline
from src.experiment_log import clear_experiments

pipeline = FPLPipeline('data', n_sims=5000)
pipeline.load_data()
pipeline.compute_features()

# Clear old experiments (old MAE was not using 25/26 actuals with bonus)
clear_experiments('data')
print("Cleared old experiment history (incompatible MAE calculation)")

# Full hyperparameter tuning (all models, 2025/26 holdout)
pipeline.tune(
    n_iter=100,
    use_subprocess=True,
    test_season='2025/2026',
    description='baseline: full tune, 5k sims, 25/26 holdout inc-bonus MAE',
    test_start_gw=20,
)

# Train final models on all data
pipeline.train()

# Predict
predictions = pipeline.predict(gameweek=31, season='2025/2026')

LOADING DATA
Loading player stats from: player_stats.csv
  Loaded 83,766 player-match records
  Seasons: ['2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Loaded 2,962 fixtures
Filtered to seasons: ['2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Current season (2025/2026): 521 active players
Final dataset: 63,493 records
Fetching actual FPL points for GW 1-30 (30 GWs)...
  Cache has 30 GWs already
  Cached to data\fpl_actual_points.csv
  Total: 23,086 player-gameweek records
  FPL card merge: 5,456 rows matched, 5,456 with card data, 395 total yellows, 42,753 DGW rows excluded

COMPUTING FEATURES
Computing rolling features...


c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:136: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'fouls_committed_per90_roll{window}'] = df.groupby('player_id')['fouls_committed_per90'].transform(
c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:136: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'fouls_committed_per90_roll{window}'] = df.groupby('player_id')['fouls_committed_per90'].transform(
c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\features.py:136: PerformanceWarning: D

  Computed 288 rolling/lifetime features
Cleared old experiment history (incompatible MAE calculation)

TUNING HYPERPARAMETERS WITH HOLDOUT TEST SET

Data split (season=2025/2026):
  Train: 60,160 samples (['2020/2021', '2021/2022', '2022/2023', '2023/2024', '2024/2025', '2025/2026'])
  Test:  3,320 samples (['2025/2026'])
    2025/2026: GW20.0-GW31.0

------------------------------------------------------------
PHASE 1: Hyperparameter + Feature Selection Tuning (5-fold CV)
------------------------------------------------------------


C:\Users\dpfin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Tuning MINUTES (two-stage, 100 trials each)...
  Total played: 60,160

  [1/3] StarterClassifier (75 features, 71.5% starters)
    Pre-computing feature rankings...


...


Model           Metric          Test Value   MAE         
------------------------------------------------------
MINUTES         Huber Loss      143.6596     17.7111     
  Classifier    AUC             0.8070       LogLoss 0.7720
  Starter Reg   MAE             11.9767     
  Sub Reg       MAE             30.6416     
GOALS           Poisson Dev     0.3731       0.1518      
ASSISTS         Poisson Dev     0.3320       0.1201      
DEFCON          Poisson Dev     2.3099       2.7593      
SAVES           MAE             1.4247       1.4247      
CLEAN_SHEET     Poisson Dev     1.0981       0.8759      
------------------------------------------------------
FPL POINTS      ex-bonus MAE    1.7376      
                inc-bonus MAE   1.8753      
BONUS           MAE             0.2819      

(Lower is better for all metrics)
ex-bonus: FPL points excluding bonus on both sides
inc-bonus: FPL points with bonus on both sides (pred bonus vs actual bonus)

Experiment logged: 20260320_152928_

C:\Users\dpfin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\xgboost\core.py:158: UserWarning: [15:29:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


  StarterMinutesModel: 45,343 samples, mean=85.9
  SubMinutesModel: 18,137 samples, mean=22.0
  Combined MAE: 15.2
Training CleanSheetModel (Goals Against) on 4,006 team-matches (50 features, tuned selection)...
  Avg goals conceded: 1.375
  Actual CS rate: 27.5%
  Prior lambda range: 0.30 - 4.00
  MAE (goals against): 0.928
  Predicted CS prob (mean): 30.3%
  Predicted 2+ conceded prob (mean): 37.4%
  Predicted CS prob range: 2.6% - 72.9%

Generating leak-free predicted team goals (TimeSeriesSplit OOF)...


c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\pipeline.py:279: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.df['opponent_norm'] = self.df['opponent'].apply(normalize_name)


  Mapped pred_team_goals to 63,493/63,493 player rows
  Mean pred_team_goals: 1.393
Training GoalsModel on 63,480 samples (75 features, tuned selection)...
  Target mean: 0.0951
  MAE: 0.1567
Training AssistsModel on 63,480 samples (55 features, tuned selection)...
  Target mean: 0.0678
  MAE: 0.1264
Training DefconModel on 63,480 samples (19 features, tuned selection)...
  Target mean: 5.8964
  MAE: 2.3160
  NB dispersion r=16.08 (var/mean ≈ 1.39x)
Training BonusModel (Monte Carlo, 5000 sims)...
  Estimating baseline BPS from stats (no actual BPS data)
Training BaselineBPSModel on 45343 samples...
  Features used: 32
  Avg baseline BPS: 18.7
  MAE: 0.68
  Loaded FPL availability for 1612 players
Training CardsModel on 63,480 samples (50 features)...
  Yellow cards: 395/63,480 (0.6%)
  LogLoss: 0.0195
Training SavesModel on 4,397 GK samples (37 features, tuned selection)...
  Mean saves/90: 2.99
  MAE: 1.465

PREDICTING GW31 (2025/2026)
Found 520 active players with historical data
  F

In [4]:
# =============================================================================
# STEP 3: View Top Players
# =============================================================================
# Top 20 by expected points with full prediction breakdown
display_cols = [
    'player_name', 'team', 'fpl_position', 'opponent', 'is_home',
    'pred_minutes', 'pred_exp_goals', 'pred_exp_assists', 
    'pred_cs_prob', 'pred_2plus_conceded', 'pred_4plus_conceded',
    'pred_defcon_prob', 'pred_exp_saves',
    'pred_yellow_prob', 'pred_red_prob',
    'pred_bonus', 'exp_total_pts'
]
available_cols = [c for c in display_cols if c in predictions.columns]
predictions.loc[predictions['fpl_position']!="GK"].nlargest(20, 'exp_total_pts')[available_cols].round(2)

,player_name,team,fpl_position,opponent,is_home,pred_minutes,pred_exp_goals,pred_exp_assists,pred_cs_prob,pred_2plus_conceded,pred_4plus_conceded,pred_defcon_prob,pred_exp_saves,pred_yellow_prob,pred_red_prob,pred_bonus,exp_total_pts
224,Cole Palmer,Chelsea,MID,Everton,0,83.070000,0.52,0.23,0.32,0.31,0.03,0.05,0.0,0.00,0.00,0.79,6.49
49,Harry Wilson,Fulham,MID,Burnley,1,85.870003,0.37,0.24,0.37,0.26,0.02,0.03,0.0,0.00,0.00,0.47,5.46
42,Bruno Fernandes,Manchester United,MID,Bournemouth,0,89.290001,0.30,0.26,0.25,0.41,0.05,0.15,0.0,0.01,0.00,0.54,5.35
142,Bruno Guimaraes,Newcastle United,MID,Sunderland,1,89.709999,0.24,0.24,0.35,0.28,0.02,0.27,0.0,0.00,0.01,0.45,5.24
221,Kevin Schade,Brentford,MID,Leeds,0,86.040001,0.37,0.15,0.27,0.37,0.04,0.04,0.0,0.00,0.00,0.54,5.16
32,Raul Jiménez,Fulham,FWD,Burnley,1,79.870003,0.45,0.14,0.37,0.26,0.02,0.01,0.0,0.00,0.00,0.94,5.16
245,Malick Thiaw,Newcastle United,DEF,Sunderland,1,89.650002,0.16,0.06,0.35,0.28,0.02,0.31,0.0,0.02,0.00,0.28,5.11
263,Lewis Hall,Newcastle United,DEF,Sunderland,1,85.750000,0.05,0.10,0.35,0.28,0.02,0.16,0.0,0.00,0.00,0.99,5.02
248,Florian Wirtz,Liverpool,MID,Brighton,0,88.360001,0.30,0.21,0.35,0.29,0.02,0.06,0.0,0.07,0.00,0.51,5.01
129,Cody Gakpo,Liverpool,MID,Brighton,0,82.239998,0.31,0.19,0.35,0.29,0.02,0.03,0.0,0.00,0.00,0.47,4.96


In [5]:
# =============================================================================
# Interactive Distribution (D3.js) — opens in browser
# =============================================================================
from src.viz import generate_distribution_html, generate_mobile_html
import webbrowser, os

# Get eval metrics from training
viz_metrics = pipeline.get_viz_metrics()

# Desktop version (ridge plot)
html_path = generate_distribution_html(
    predictions,
    pipeline.last_simulations,
    output_path='distributions.html',
    top_n=100,
    gameweek=31,
    metrics=viz_metrics,
)

# Mobile version (card layout)
mobile_path = generate_mobile_html(
    predictions,
    pipeline.last_simulations,
    output_path='distributions_mobile.html',
    top_n=100,
    gameweek=31,
    metrics=viz_metrics,
)

# Save full run (predictions, simulations, params, metrics)
run_dir = pipeline.save_run(
    predictions,
    gameweek=31,
    season='2025/2026',
    description='baseline: full tune, 5k sims, 25/26 holdout inc-bonus MAE',
)

webbrowser.open('file://' + os.path.realpath(html_path))

Distribution visualization saved to: distributions.html
Mobile distribution visualization saved to: distributions_mobile.html

Run saved to: data\runs\gw31_20260320_152954
  predictions.csv: 347 players
  simulations: 5000 sims
  tuned_params.json: 6 models
  FPL MAE: ex-bonus=1.7376, inc-bonus=1.8753


True

In [6]:
# =============================================================================
# FPL Points Breakdown: Predicted vs Actual by Category
# Shows where the model over/under-predicts in FPL points terms
# =============================================================================
breakdown = pipeline.points_breakdown()
display(breakdown.style.format({
    'pred_value_avg': '{:.4f}',
    'pred_pts_avg': '{:.4f}',
    'actual_value_avg': '{:.4f}',
    'actual_pts_avg': '{:.4f}',
    'pts_diff': '{:+.4f}',
    'abs_pts_diff': '{:.4f}',
}, na_rep='—').set_caption('FPL Points Breakdown: Predicted vs Actual (test set averages per player-match)'))

c:\Users\dpfin\OneDrive\Desktop\projecting_fpl_v2\src\pipeline.py:2163: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, totals], ignore_index=True)


,category,pred_value_avg,pred_pts_avg,actual_value_avg,actual_pts_avg,pts_diff,abs_pts_diff
0,Appearance,67.8457,1.6789,66.0449,1.6928,-0.0139,0.0139
1,Goals,0.0928,0.4317,0.0861,0.3955,+0.0362,0.0362
2,Assists,0.0693,0.2078,0.0627,0.1880,+0.0199,0.0199
3,Clean Sheet,0.2098,0.3342,0.2271,0.4843,-0.1501,0.1501
4,Goals Conceded,1.3953,-0.1502,1.2393,-0.1346,-0.0156,0.0156
5,Saves,2.9096,0.0657,2.6667,0.0380,+0.0278,0.0278
6,Defcon,0.2154,0.2114,0.2290,0.2247,-0.0133,0.0133
7,Yellow Cards,0.0301,-0.0301,0.0422,-0.0422,+0.0120,0.0120
8,Red Cards,0.0027,-0.0080,0.0021,-0.0063,-0.0017,0.0017
9,Bonus,0.2205,0.2205,0.0873,0.0873,+0.1331,0.1331
